# Load & Process Spotify Playlist Dataset

Our main dataset is from the Spotify Million Playlist Dataset Challenge. 

The link to the dataset can be found here: https://www.aicrowd.com/challenges/spotify-million-playlist-dataset-challenge

In [ ]:
import pandas as pd
import glob
import json
from concurrent.futures import ThreadPoolExecutor
from collections import defaultdict

In [ ]:
# ----- 1.1 Read JSON Playlists in Parallel -----
def read_json_file(filename):
    with open(filename, 'r') as file:
        data = json.load(file)
    return data['playlists']

json_path = '../spotify_million_playlist_dataset/data/*.json'
all_files = glob.glob(json_path)
with ThreadPoolExecutor(max_workers=10) as executor:
    results = executor.map(read_json_file, all_files)

data_list = [item for sublist in results for item in sublist]
raw_df = pd.DataFrame(data_list)
print("Loaded raw playlists. Shape:", raw_df.shape)

In [ ]:
# ----- 1.2 Filter and Sort Playlists -----
# (Assumes the column 'collaborative' is stored as the string 'false' for non-collaborative playlists.
# Adjust if it is boolean.)
filtered_df = raw_df[raw_df['collaborative'] == 'false'].copy()
filtered_df['modified_at'] = pd.to_datetime(filtered_df['modified_at'], unit='s')
sorted_df = filtered_df.sort_values(by='modified_at', ascending=False)
print("After filtering and sorting. Shape:", sorted_df.shape)

In [ ]:
# ----- 1.3 Select Top Playlists -----
num_of_playlist = 20000
final_playlists_df = sorted_df.head(num_of_playlist)
print("Final playlists (top 20k) shape:", final_playlists_df.shape)

In [ ]:
# ----- 1.4 Build Track Details DataFrame -----
# Build a dictionary keyed by track_uri that collects:
#   - the playlist IDs (pid) in which the track appears
#   - the track information (stored only once)
track_details_dict = defaultdict(lambda: {'inside_playlists': [], 'track_info': {}})
for _, playlist in final_playlists_df.iterrows():
    pid = playlist['pid']
    for track in playlist['tracks']:
        track_uri = track['track_uri']
        track_details_dict[track_uri]['inside_playlists'].append(pid)
        # Store the track metadata once (if not already stored)
        if not track_details_dict[track_uri]['track_info']:
            track_details_dict[track_uri]['track_info'] = track

track_details_df = pd.DataFrame([
    {'track_uri': track_uri, **details['track_info'], 'inside_playlists': details['inside_playlists']}
    for track_uri, details in track_details_dict.items()
])
print("Raw track details shape:", track_details_df.shape)

# Audio Features Merging

We have used a dataset from Kaggle that contained the audio features specifically retrieved for the tracks inside the Spotify Million Playlist Dataset Challenge.

The link to the dataset can be found here: https://www.kaggle.com/datasets/krishsharma0413/2-million-songs-from-mpd-with-audio-features

In [ ]:
import pandas as pd
import glob
import os
import pyarrow.parquet as pq

Since the primary key for the dataset is also the track_uri, we just had to do a inner join.

In [ ]:
# ----- 2.1 Load and Merge Audio Features -----
def load_audio_features(audio_features_file):
    columns = [
        'track_uri', 'danceability', 'energy', 'key', 'loudness', 
        'mode', 'speechiness', 'acousticness', 'instrumentalness', 
        'liveness', 'valence', 'tempo', 'time_signature'
    ]
    return pd.read_parquet(audio_features_file, columns=columns)

audio_features_file = "audio_features_for_all_songs.parquet"
audio_df = load_audio_features(audio_features_file)

# Merge audio features with raw track details from Section 1
# (Assume track_details_df is available from Section 1)
tracks_audio_df = pd.merge(track_details_df, audio_df, on="track_uri", how="inner")
print("After merging audio features:", tracks_audio_df.shape)

# Let the enriched DataFrame start with the audio merge
tracks_enriched_df = tracks_audio_df.copy()

# Lyrics Merging

We have tried searching datasets from Kaggle so as to retrieve the lyrics for each of the tracks. 

However, unlike the audio features, there was no dataset containing lyrics that were created specifically for the tracks inside the Spotify Million Playlist Dataset Challenge.

As such, we used multiple datasets from Kaggle.

The link to the datasets can be found here: 

- Source 1: https://www.kaggle.com/datasets/tnssivani/spotify-960k-lyrics-dataset-with-timestamps

- Source 2: https://www.kaggle.com/datasets/notshrirang/spotify-million-song-dataset

- Source 3: https://www.kaggle.com/datasets/edenbd/150k-lyrics-labeled-with-spotify-valence

- Source 4: https://www.kaggle.com/datasets/saurabhshahane/spotgen-music-dataset

- Source 5: https://www.kaggle.com/datasets/nikhilnayak123/5-million-song-lyrics-dataset

In [ ]:
# ----- 2.2 Define Helper Function for Lyrics Merging -----
def merge_lyrics_by_id(enriched, csv_filepath, csv_cols=["id", "lyrics"], rename_mapping=None):
    """
    Merge lyrics from a CSV that provides a track identifier.
    The function removes the "spotify:track:" prefix from the enriched DataFrame,
    then merges on that cleaned ID and the CSV key.
    """
    csv_df = pd.read_csv(csv_filepath, usecols=csv_cols, dtype=str)
    if rename_mapping:
        csv_df.rename(columns=rename_mapping, inplace=True)
    enriched = enriched.copy()
    enriched["clean_track_id"] = enriched["track_uri"].str.replace("spotify:track:", "", regex=False)
    merged = pd.merge(enriched, csv_df, left_on="clean_track_id", right_on="id", how="left")
    merged.drop(columns=["clean_track_id", "id"], inplace=True)
    return merged

def merge_lyrics_by_artist_song(enriched, csv_filepath, csv_cols, rename_mapping=None):
    """
    Merge lyrics from a CSV that provides song lyrics along with artist and song title.
    This function expects the enriched DataFrame to have columns 'artist_name' and 'track_name'.
    """
    csv_df = pd.read_csv(csv_filepath, usecols=csv_cols, dtype=str)
    if rename_mapping:
        csv_df.rename(columns=rename_mapping, inplace=True)
    merged = pd.merge(enriched, csv_df, left_on=["artist_name", "track_name"], right_on=["artist", "song"], how="left")
    merged.drop(columns=["artist", "song"], inplace=True)
    return merged

# For each merge, we update the tracks_enriched_df only for tracks that are missing lyrics.
def update_missing_lyrics(enriched, merge_func, **merge_kwargs):
    """Apply a merge function only on rows missing lyrics, and then recombine."""
    without_lyrics = enriched[enriched["lyrics"].isna()].copy()
    if without_lyrics.empty:
        # All tracks already have lyrics.
        return enriched
    merged = merge_func(without_lyrics, **merge_kwargs)
    # Combine the updated rows with the ones that already had lyrics.
    with_lyrics = enriched.drop(enriched[enriched["lyrics"].isna()].index)
    combined = pd.concat([with_lyrics, merged], ignore_index=True)
    return combined

In [ ]:
# ----- 2.3 Enrich with External Lyrics Sources -----
# Source 1: songs_with_attributes_and_lyrics.csv (merging on track ID)
tracks_enriched_df = update_missing_lyrics(
    tracks_enriched_df,
    merge_func=merge_lyrics_by_id,
    csv_filepath="songs_with_attributes_and_lyrics.csv",
    csv_cols=["id", "lyrics"],
    rename_mapping=None
)
num_with = tracks_enriched_df.dropna(subset=["lyrics"]).shape[0]
num_without = tracks_enriched_df[tracks_enriched_df["lyrics"].isna()].shape[0]
print(f"After source 1 (attributes): with lyrics: {num_with}, without lyrics: {num_without}")

# Source 2: spotify_millsongdata.csv (merging on artist and song)
tracks_enriched_df = update_missing_lyrics(
    tracks_enriched_df,
    merge_func=merge_lyrics_by_artist_song,
    csv_filepath="spotify_millsongdata.csv",
    csv_cols=["artist", "song", "text"],
    rename_mapping={"text": "lyrics"}
)
num_with = tracks_enriched_df.dropna(subset=["lyrics"]).shape[0]
num_without = tracks_enriched_df[tracks_enriched_df["lyrics"].isna()].shape[0]
print(f"After source 2 (millsongdata): with lyrics: {num_with}, without lyrics: {num_without}")

# Source 3: labeled_lyrics_cleaned.csv (merging on artist and song)
tracks_enriched_df = update_missing_lyrics(
    tracks_enriched_df,
    merge_func=merge_lyrics_by_artist_song,
    csv_filepath="labeled_lyrics_cleaned.csv",
    csv_cols=["artist", "song", "seq"],
    rename_mapping={"seq": "lyrics"}
)
num_with = tracks_enriched_df.dropna(subset=["lyrics"]).shape[0]
num_without = tracks_enriched_df[tracks_enriched_df["lyrics"].isna()].shape[0]
print(f"After source 3 (labeled lyrics): with lyrics: {num_with}, without lyrics: {num_without}")

# Source 4: spotify_tracks.csv (merging on track_uri and uri)
def merge_lyrics_by_uri(enriched, csv_filepath, csv_cols=["uri", "lyrics"]):
    csv_df = pd.read_csv(csv_filepath, usecols=csv_cols, dtype=str)
    merged = pd.merge(enriched, csv_df, left_on="track_uri", right_on="uri", how="left")
    merged.drop(columns=["uri"], inplace=True)
    return merged

tracks_enriched_df = update_missing_lyrics(
    tracks_enriched_df,
    merge_func=merge_lyrics_by_uri,
    csv_filepath="spotify_tracks.csv",
    csv_cols=["uri", "lyrics"]
)
num_with = tracks_enriched_df.dropna(subset=["lyrics"]).shape[0]
num_without = tracks_enriched_df[tracks_enriched_df["lyrics"].isna()].shape[0]
print(f"After source 4 (spotify_tracks): with lyrics: {num_with}, without lyrics: {num_without}")

In [ ]:
# Now tracks_enriched_df has been updated through all external sources.
print("Final enriched tracks shape (after audio and lyrics merging):", tracks_enriched_df.shape)

# Popularity & Release Metadata Enrichment

We have also used the Spotify API to retrieve the popularity scores and the album release date for each track using the track_uri.

However, some tracks have been removed from Spotify and we were thus unable to retrieve for those tracks.

We abided by Spotify API's limits with timeouts in between.

In [ ]:
import os
import pandas as pd
import pickle
import requests
import spotipy
import time
import logging
from spotipy.oauth2 import SpotifyClientCredentials
from dotenv import load_dotenv
from tqdm import tqdm

In [ ]:
# 3.1: Environment Setup and Spotify Authentication
logging.basicConfig(
    filename="spotify_extraction.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
)
load_dotenv()
SPOTIPY_CLIENT_ID = os.getenv("SPOTIPY_CLIENT_ID")
SPOTIPY_CLIENT_SECRET = os.getenv("SPOTIPY_CLIENT_SECRET")
print("Spotify Client ID:", SPOTIPY_CLIENT_ID)
if not SPOTIPY_CLIENT_ID or not SPOTIPY_CLIENT_SECRET:
    raise Exception("Missing Spotify credentials. Check your .env file.")
sp = spotipy.Spotify(auth_manager=SpotifyClientCredentials(
    client_id=SPOTIPY_CLIENT_ID,
    client_secret=SPOTIPY_CLIENT_SECRET
))


In [ ]:
# 3.2: Extract Unique Track URIs and Album Names from Enriched Tracks
unique_track_uris = tracks_enriched_df["track_uri"].dropna().unique().tolist()
unique_album_names = tracks_enriched_df["album_name"].dropna().unique().tolist()
print("Unique tracks:", len(unique_track_uris), "| Unique albums:", len(unique_album_names))

In [ ]:
# 3.3: Caching Setup and Helper Functions
TRACK_PROGRESS_FILE = "track_popularity_progress.pkl"
ALBUM_PROGRESS_FILE = "album_release_progress.pkl"

def load_progress(file_path):
    if os.path.exists(file_path) and os.path.getsize(file_path) > 0:
        with open(file_path, "rb") as f:
            return pickle.load(f)
    return {}

def extract_track_id(uri):
    parts = uri.split(":")
    return parts[2] if len(parts) == 3 else uri

track_ids = [extract_track_id(uri) for uri in unique_track_uris]
track_popularity_dict = load_progress(TRACK_PROGRESS_FILE)
album_release_dict = load_progress(ALBUM_PROGRESS_FILE)
logging.info(f"Total unique tracks: {len(track_ids)} | Already processed: {len(track_popularity_dict)}")
logging.info(f"Total unique albums: {len(unique_album_names)} | Already processed: {len(album_release_dict)}")

In [ ]:
# 3.4: Define Functions to Fetch Data with Rate Limit Handling
def fetch_track_popularity(track_batch):
    while True:
        try:
            response = sp.tracks(track_batch)
            return {track["id"]: track["popularity"] for track in response["tracks"] if track}
        except spotipy.SpotifyException as e:
            if e.http_status == 429:
                retry_after = int(e.headers.get("Retry-After", 5))
                logging.warning(f"Rate limit hit for tracks. Retrying in {retry_after} seconds...")
                time.sleep(retry_after)
            else:
                logging.error(f"Error fetching track popularity: {e}")
                return {}

def fetch_album_release(album_batch):
    results = {}
    for album_name in album_batch:
        while True:
            try:
                search_results = sp.search(q=f"album:{album_name}", type="album", limit=1)
                items = search_results["albums"]["items"]
                if items:
                    album_id = items[0]["id"]
                    album_data = sp.album(album_id)
                    results[album_data["name"]] = album_data["release_date"]
                else:
                    results[album_name] = None
                break
            except spotipy.SpotifyException as e:
                if e.http_status == 429:
                    retry_after = int(e.headers.get("Retry-After", 5))
                    logging.warning(f"Rate limit hit for album '{album_name}'. Retrying in {retry_after} seconds...")
                    time.sleep(retry_after)
                else:
                    logging.error(f"SpotifyException for album '{album_name}': {e}")
                    results[album_name] = None
                    break
            except requests.exceptions.ReadTimeout:
                logging.error(f"ReadTimeout for album '{album_name}'. Skipping.")
                results[album_name] = None
                break
            except requests.exceptions.RequestException as re:
                logging.error(f"RequestException for album '{album_name}': {re}")
                results[album_name] = None
                break
            except Exception as ex:
                logging.error(f"Unexpected error for album '{album_name}': {ex}")
                results[album_name] = None
                break
    return results

In [ ]:
# 3.5: Process Track Popularity in Batches (50 IDs per batch)
track_batches = [track_ids[i:i+50] for i in range(0, len(track_ids), 50)]
remaining_track_batches = [batch for batch in track_batches if not set(batch).issubset(track_popularity_dict.keys())]
logging.info(f"Remaining track batches: {len(remaining_track_batches)}")
for batch in tqdm(remaining_track_batches, desc="Fetching Track Popularity"):
    track_popularity_dict.update(fetch_track_popularity(batch))
    if len(track_popularity_dict) % 500 < 50:
        with open(TRACK_PROGRESS_FILE, "wb") as f:
            pickle.dump(track_popularity_dict, f)
        logging.info(f"Saved {len(track_popularity_dict)} track popularity records.")
with open(TRACK_PROGRESS_FILE, "wb") as f:
    pickle.dump(track_popularity_dict, f)
logging.info(f"Final track popularity records: {len(track_popularity_dict)}.")

In [ ]:
# 3.6: Process Album Release Dates in Batches (20 per batch)
album_batches = [unique_album_names[i:i+20] for i in range(0, len(unique_album_names), 20)]
remaining_album_batches = [batch for batch in album_batches if not set(batch).issubset(album_release_dict.keys())]
logging.info(f"Remaining album batches: {len(remaining_album_batches)}")
for batch in tqdm(remaining_album_batches, desc="Fetching Album Release Dates"):
    album_release_dict.update(fetch_album_release(batch))
    if len(album_release_dict) % 100 < 20:
        with open(ALBUM_PROGRESS_FILE, "wb") as f:
            pickle.dump(album_release_dict, f)
        logging.info(f"Saved {len(album_release_dict)} album release records.")
with open(ALBUM_PROGRESS_FILE, "wb") as f:
    pickle.dump(album_release_dict, f)
logging.info(f"Final album release records: {len(album_release_dict)}.")

In [ ]:
# 3.7: Merge Fetched Data into the Enriched Tracks DataFrame
# We update tracks_enriched_df (which was created in Section 2/3) by adding Spotify metadata.
tracks_enriched_df = tracks_enriched_df.copy()  # Ensure we work on a copy
tracks_enriched_df["track_popularity"] = tracks_enriched_df["track_uri"].apply(lambda uri: track_popularity_dict.get(extract_track_id(uri)))
tracks_enriched_df["album_release_date"] = tracks_enriched_df["album_name"].map(album_release_dict)
print("Tracks enriched with popularity and album release metadata. Shape:", tracks_enriched_df.shape)